# Day 73: Lab - Sentiment Analysis Mini-Project

## Project Goal
This lab puts together everything from the last two notebooks - tokenizing text, padding sequences, and an Embedding layer - and adds an LSTM to build a real (if tiny) sentiment classifier: given a short product review, predict whether it's positive (1) or negative (0). The dataset here is intentionally small and hand-written just to demonstrate the full pipeline end to end; a real sentiment analysis project would use thousands or millions of labeled reviews. The code below just defines the raw text reviews and their corresponding labels (1 = positive, 0 = negative) - the actual text processing and modeling happens in the next steps.</cell id="cell-1">


In [ ]:
# pip install tensorflow
reviews = [
    'this product is amazing and works great',
    'terrible experience, would not recommend',
    'I love it, best purchase ever',
    'waste of money, very disappointed',
    'excellent quality and fast delivery',
    'awful service and broken item',
    'highly satisfied with this purchase',
    'poor quality, stopped working in a week'
]
labels = [1, 0, 1, 0, 1, 0, 1, 0]  # 1 = positive sentiment, 0 = negative sentiment, aligned index-for-index with reviews

## Step 1: Tokenize & Pad
This is the standard NLP preprocessing pipeline: `Tokenizer` builds a word -> integer vocabulary from the reviews, `texts_to_sequences` turns each review into a list of those integers in order, and `pad_sequences` forces every sequence to the same fixed length (`maxlen=10`) by padding shorter ones with zeros and truncating longer ones. The result, `X`, is a 2D array (number of reviews x 10) of integers that an Embedding layer can consume directly, and `y` is simply the labels converted to a numpy array for training. Getting text into this uniform numeric shape is a prerequisite for feeding it into any Keras model, whether it uses an Embedding+LSTM combo like this lab or an Embedding+Dense combo like the previous notebook.</cell id="cell-3">


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

tokenizer = Tokenizer()
tokenizer.fit_on_texts(reviews)  # build vocabulary (word -> integer index) from the review texts
sequences = tokenizer.texts_to_sequences(reviews)  # convert each review into a list of word-index integers
X = pad_sequences(sequences, maxlen=10)  # pad/truncate every review to exactly 10 tokens
y = np.array(labels)

## Step 2: Build the Model
This architecture chains together the two sequence-modeling building blocks from the last two notebooks: the Embedding layer converts each word-index into a dense 8-dimensional vector that captures word meaning, and the LSTM then reads that sequence of vectors one word at a time, using its gated memory to build up an understanding of the review's overall sentiment (this is a better fit than the earlier Flatten+Dense approach for embeddings, since LSTM actually respects word order rather than just concatenating vectors). Unlike the word-embeddings notebook, here we skip Flatten entirely - LSTM naturally reduces a sequence down to a single summary vector (its final hidden state), which then feeds directly into a Dense(1, sigmoid) layer that outputs a probability of the review being positive. `binary_crossentropy` is the correct loss since this is a two-class (positive/negative) classification problem with a single sigmoid output.</cell id="cell-5">


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

vocab_size = len(tokenizer.word_index) + 1  # +1 for the reserved padding index (0)
model = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=8, input_length=10),  # word index -> 8-dim learned vector
    layers.LSTM(16),  # reads the sequence of word vectors, outputs one summary vector per review
    layers.Dense(1, activation='sigmoid')  # positive-sentiment probability
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## Step 3: Train & Test on New Text
With only 8 training examples, this model will essentially memorize the training reviews rather than learn generalizable sentiment patterns - training for 30 epochs on such a tiny dataset is only meant to demonstrate the mechanics of the pipeline, not to produce a production-quality classifier. The real value of this cell is showing how to run inference on brand-new, unseen text: the new review has to go through the exact same preprocessing (`tokenizer.texts_to_sequences` then `pad_sequences` with the same `maxlen=10`) that the training data went through, since the model only understands the vocabulary and sequence format it was trained on. `model.predict` then returns a probability (closer to 1 = more positive, closer to 0 = more negative) rather than a hard 0/1 label, which is why we print it directly instead of rounding - in a real project you'd typically threshold this at 0.5 to get a final class prediction.</cell id="cell-7">


In [ ]:
model.fit(X, y, epochs=30, verbose=0)
new_review = ['this is a great and wonderful product']
# new text must go through the SAME tokenizer + padding (maxlen=10) as training data
new_seq = pad_sequences(tokenizer.texts_to_sequences(new_review), maxlen=10)
print('Positive probability:', model.predict(new_seq)[0][0])